# 3_SynAnalyzer_CompileXYZData.ipynb

This Python notebook reads in the CSV files generated in 2_SynAnalyzer_AnalyzeXYZs.ijm that were annotated by the user during
manual array scoring. These files were saved in SAR.Analysis. This code pulls these files and the associated Metadata files
from the Batch directory. 

In [2]:
#     *** IMPORT PACKAGES **
import datetime
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
import glob
import seaborn as sns
import statistics
import math

In [3]:
#                   *** GET TIME OF ANALYSIS START ***
# could turn this into a module?
toa = str(datetime.datetime.today()).split()                                   #Time of analysis
today = toa[0]
now = toa[1]
timestamp = today.replace('-','')+'-'+now.replace(':','')[:6]

In [38]:
#     *** WHAT TO ANALYZE // WHERE TO GET/STORE **
# Batch analysis directory (where the new metadata sheet will be stored)
batchID = 'SynAnalyzer_DemoBatch'      
# Cutoff for automated classificaiton of subtypes
# This value was determined empirically (see Methods)
threshAutoTdT = .22

# Directories 
#   existing ones 
dirMain = '/Users/joyfranco/Partners HealthCare Dropbox/Joy Franco/JF_Shared/Data/WSS/'
dirBA = dirMain+"BatchAnalysis/"+batchID+"/"
dirMD = dirBA+"Metadata/"
dirSA = dirBA+"SAR.Analysis/"

#   ones that need to be made    
dirRes = dirBA+"SAR.Results/"
dirTR = dirRes+timestamp+'/'

# Files to access
fnMDSamp = batchID+".Metadata.Samples.csv"
fnMDIms = batchID+".Metadata.Imaging.csv"

# Files to make
fnXYZ = batchID+".XYZSummary.csv"
fnBS = batchID+".BatchSummary.csv"

In [29]:
#    *** INITIALIZE RUN SPECIFIC DIRECTORY ETC FOR STORING RESULTS **
# Create directory for storing spreadsheets and summary plots for this run
if not os.path.exists(dirRes): os.mkdir(dirRes)
if not os.path.exists(dirTR): os.mkdir(dirTR)

In [30]:
#    *** LOAD MD SHEETS **
# Sample metadata sheet that include animal and condition info
dfSamps = pd.read_csv(dirMD+fnMDSamp)
#dfSamps.drop('Unnamed: 0', axis=1, inplace=True)
# Imaging metadata sheet that includes frequency and NoHC info
dfIms = pd.read_csv(dirMD+fnMDIms)

In [31]:
#   *** GET LIST OF ALL XYZ FILES TO COMPILE ***
allFiles = os.listdir(dirSA)
xyzFiles = []
for f in allFiles:
    if "XYZ" in f:
        xyzFiles.append(f)
xyzFiles.sort()
xyzFiles

['WSS_030.01.T2.01.Zs.4C.XYZ.PostSyn.csv',
 'WSS_030.01.T2.01.Zs.4C.XYZ.PreSyn.csv',
 'WSS_031.03.T3.01.Zs.4C.XYZ.PostSyn.csv',
 'WSS_031.03.T3.01.Zs.4C.XYZ.PreSyn.csv']

In [47]:
#   *** COMPILE XYZ DATA ***
# Initialize df for compiling all xyz data
dfXYZAll = pd.DataFrame()
for f in xyzFiles:
    # Get the metadata from the filename
    samp = f[0:10]
    imName = f[0:22]
    surfType = f[27:(len(f)-4)]
    # Get the metadata for this im from md dataframes
    inSampMD = dfSamps.index[dfSamps["SampleID"]==samp].tolist()[0]
    anID = dfSamps.loc[inSampMD]["AnimalID"]
    group = dfSamps.loc[inSampMD]["Group"]
    inImMD = dfIms.index[dfIms["ImageName"]==imName].tolist()[0]
    freq = dfIms.loc[inImMD]["Frequency"]
    noHC = dfIms.loc[inImMD]["HairCellsReconstructed"]
    # Load the file
    df = pd.read_csv(dirSA+f)
    df["SurfType"] = surfType
    df["SampleID"] = samp
    df["ImageName"] = imName
    df["AnimalID"] = anID
    df["Group"] = group
    df["Frequency"] = freq
    df["NoHCRecon"] = noHC
    
    #Calculate the image maximum normalized tdT intensity
    maxCh4Int = df["uIntCh_4"].max() 
    df["MaxNormTdTom"] = df["uIntCh_4"]/maxCh4Int
 
    # Setup Autoclassified Subtype based on image maximum normalized tdTOM intensity
    if surfType == "PostSyn":
        df["AutoTdTStatus"] = "TBD"
        for index, row in df.iterrows():
            tdTomVal = df.loc[index]["MaxNormTdTom"]
            if (tdTomVal >= threshAutoTdT):
                df.at[index,"AutoTdTStatus"] = "Positive"
            else:
                df.at[index,"AutoTdTStatus"] = "Negative"
    else:
        df["AutoTdTStatus"] = "NA"
    
    # Clear up any empty entries
    df = df.fillna('')
    
    # Add it to the main df
    dfXYZAll = pd.concat([dfXYZAll,df])

dfXYZAll.reset_index(inplace=True)
dfXYZAll.drop(["index"], axis=1, inplace=True)
dfXYZAll.to_csv(dirTR+fnXYZ) 

In [48]:
#   *** CALCULATE THE CONTROL MEDIAN NORMALIZED VOLUMES ***
# This calculates a normalized PSD volume using the median volume from all control PSDs
ctrlMedianVol = statistics.median(dfXYZAll[dfXYZAll["Group"]=="Sham"]["Volume_um3"])
dfXYZAll["CtrlMedNormVol"] = dfXYZAll["Volume_um3"]/ctrlMedianVol
dfXYZAll.to_csv(dirTR+fnXYZ)

In [49]:
#   *** GENERATE IMAGE SUMMARY SHEET ***
# Initialize df for compiling all image data
dfImsAll = pd.DataFrame()
for im in dfXYZAll["ImageName"].unique():
    samp = im[0:10]
    # Get metadata associated with each image
    inSampMD = dfSamps.index[dfSamps["SampleID"]==samp].tolist()[0]
    anID = dfSamps.loc[inSampMD]["AnimalID"]
    group = dfSamps.loc[inSampMD]["Group"]
    inImMD = dfIms.index[dfIms["ImageName"]==im].tolist()[0]
    freq = dfIms.loc[inImMD]["Frequency"]
    noHC = float(dfIms.loc[inImMD]["HairCellsReconstructed"])
    # Setup entry in df for this image
    df = pd.DataFrame({'ImageName': im, 
                       'SampleID':samp, 
                       'AnimalID':anID,
                       'Group':group,
                       'Freq':[freq],
                       'NoHCRecon':[noHC]
                       })
    # Get surface types associated with this im
    surfs = dfXYZAll[dfXYZAll["ImageName"] == im]["SurfType"].unique()
    df["SurfacesAvailable"] = str(surfs)

    # Get surface types associated with this im
    surfs = dfXYZAll[dfXYZAll["ImageName"] == im]["SurfType"].unique()
    df["SurfacesAvailable"] = str(surfs)
    # Iterate through the available surface types
    for surf in surfs:
        # CALCULATE SYNAPSE INFORMATION
        df[surf+"_PairedSynapseIndex"] = len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                    (dfXYZAll["SurfType"] == surf) &
                                    (dfXYZAll["SynapseStatus"] == "Synapse")])/noHC
        df[surf+"_UnpairedIndex"] = len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                    (dfXYZAll["SurfType"] == surf) &
                                    (dfXYZAll["SynapseStatus"] == "Unpaired")])/noHC
       
        if surf=="PostSyn":
            # CALCULATE INFORMATION ON TERMINAL SCORING
            df[surf+"_TdTPosPairedSynIndex"] = len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                        (dfXYZAll["SurfType"] == surf) &
                                        (dfXYZAll["SynapseStatus"] == "Synapse")&
                                        (dfXYZAll["TerminalStatus"] == "Positive")])/noHC
            df[surf+"_TdTPosUnpairedSynIndex"] = len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                        (dfXYZAll["SurfType"] == surf) &
                                        (dfXYZAll["SynapseStatus"] == "Unpaired")&
                                        (dfXYZAll["TerminalStatus"] == "Positive")])/noHC
            try:
                df[surf+"_TdTPosPairedSynProp"] = len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["TerminalStatus"] == "Positive")])/len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")])
            except:
                df[surf+"_TdTPosPairedSynprop"] = 0
    
            try:
                df[surf+"_TdTPosUnpairedSynProp"] = (len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Unpaired")&
                                            (dfXYZAll["TerminalStatus"] == "Positive")])/len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Unpaired")]))
            except:
                df[surf+"_TdTPosUnpairedSynProp"] = 0
        
            
            try:
                df[surf+"_AutoTdTPosProp"] = (len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["AutoTdTStatus"] == "Positive")])/len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")]))
            except:
                df[surf+"_AutoTdTPosProp"] = 0

            try:
                df[surf+"_PropTermUncSyns"] = len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["TerminalStatus"] == "Uncertain")])/len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")])
            except:
                df[surf+"_PropTermUncSyns"] = 0
    
            try:
                df[surf+"_PropTermPosAndUncSyns"] = (len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["TerminalStatus"] == "Positive")])+
                                                     len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["TerminalStatus"] == "Uncertain")]))/len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")])
            except:
                df[surf+"_PropTermPosAndUncSyns"] = 0
    
            try:
                df[surf+"_PropTermNegAndUncSyns"] = (len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["TerminalStatus"] == "Negative")])+
                                                     len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")&
                                            (dfXYZAll["TerminalStatus"] == "Uncertain")]))/len(dfXYZAll[(dfXYZAll["ImageName"] == im) & 
                                            (dfXYZAll["SurfType"] == surf) &
                                            (dfXYZAll["SynapseStatus"] == "Synapse")])
            except:
                df[surf+"_PropTermNegAndUncSyns"] = 0
        
    
    # Add it to the main df
    dfImsAll = pd.concat([dfImsAll,df])

dfImsAll.reset_index(inplace=True)
dfImsAll.drop(["index"], axis=1, inplace=True)
dfImsAll.to_csv(dirTR+fnBS) 